# Portfolio Bot v2 — Monthly DCA + 2x ATH Profit-Taking

**Changes vs v1:**
- Buy **every month** (no red candle filter)
- Fixed $20 BTC + $10 BNB every month (no progressive reinvestment)
- Sell **35%** only when price reaches **2x the last sell's ATH**
- No idle reserve buildup — everything goes to work

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch(s):
    url = 'https://api.binance.com/api/v3/klines'; ak = []; st = 0
    while True:
        p = {'symbol': s, 'interval': '1M', 'startTime': st, 'limit': 500}
        d = requests.get(url, params=p).json()
        if not d: break
        ak.extend(d)
        if len(d) < 500: break
        st = d[-1][0] + 1; time.sleep(0.1)
    return ak
cols = ['ts','o','h','l','c','v','ct','qv','t','tbb','tbq','ig']
btc = pd.DataFrame(fetch('BTCUSDT'), columns=cols); btc['date'] = pd.to_datetime(btc['ts'], unit='ms', utc=True)
bnb = pd.DataFrame(fetch('BNBUSDT'), columns=cols); bnb['date'] = pd.to_datetime(bnb['ts'], unit='ms', utc=True)
for c in ['o','h','l','c','v']: btc[c]=btc[c].astype(float); bnb[c]=bnb[c].astype(float)
btc = btc[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
bnb = bnb[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
s = max(btc['date'].min(), bnb['date'].min()); e = min(btc['date'].max(), bnb['date'].max())
btc = btc[(btc['date']>=s)&(btc['date']<=e)].reset_index(drop=True)
bnb = bnb[(bnb['date']>=s)&(bnb['date']<=e)].reset_index(drop=True)
print(f'BTC: {btc["date"].min().strftime("%Y-%m")} -> {btc["date"].max().strftime("%Y-%m")} ({len(btc)} mo)')
print(f'BNB: {bnb["date"].min().strftime("%Y-%m")} -> {bnb["date"].max().strftime("%Y-%m")} ({len(bnb)} mo)')

In [ ]:
# Strategy: monthly DCA + sell 35% at 2x ATH
SELL_RATIO = 0.35
SELL_MULT = 2.0  # sell when price reaches 2x the reference ATH
usdt = 0.0; injected = 0.0
bh = 0.0; bhi = 0.0; bhi_sell = 0.0  # bhi = ATH, bhi_sell = ATH at last sell
nh = 0.0; nhi = 0.0; nhi_sell = 0.0
rec = []
for i in range(len(btc)):
    b = btc.iloc[i]; n = bnb.iloc[i]
    bc = b['c']; nc = n['c']; d = b['date']; act = []
    # Inject $30
    usdt += 30.0; injected += 30.0
    # Buy $20 BTC every month
    bh += 20.0 / bc; usdt -= 20.0; act.append(f'BTC buy $20 @ {bc:.0f}')
    # Buy $10 BNB every month
    nh += 10.0 / nc; usdt -= 10.0; act.append(f'BNB buy $10 @ {nc:.0f}')
    # Update ATHs
    if bc > bhi: bhi = bc
    if nc > nhi: nhi = nc
    # BTC sell: 2x from last sell ATH
    if bh > 0.000001 and bc >= bhi_sell * SELL_MULT and bhi_sell > 0:
        s = bh * SELL_RATIO
        usdt += s * bc; bh -= s
        bhi_sell = bhi
        act.append(f'BTC sell 35% @ {bc:.0f} (2x ATH)')
    elif bhi_sell == 0:
        bhi_sell = bc  # init reference after first month
    # BNB sell: 2x from last sell ATH
    if nh > 0.000001 and nc >= nhi_sell * SELL_MULT and nhi_sell > 0:
        s = nh * SELL_RATIO
        usdt += s * nc; nh -= s
        nhi_sell = nhi
        act.append(f'BNB sell 35% @ {nc:.0f} (2x ATH)')
    elif nhi_sell == 0:
        nhi_sell = nc
    # Surplus usdt after buys accumulates (will be deployed next month)
    rec.append({'date': d, 'bc': bc, 'nc': nc,
        'bh': bh, 'bv': bh*bc, 'nh': nh, 'nv': nh*nc,
        'usdt': usdt, 'inj': injected, 'pf': bh*bc + nh*nc + usdt,
        'act': ' | '.join(act) if act else 'nothing',
        'bhi_sell': bhi_sell, 'nhi_sell': nhi_sell})
res = pd.DataFrame(rec)
buys = len(res[res['act'].str.contains('buy', na=False)])
sells = len(res[res['act'].str.contains('sell', na=False)])
print(f'Months: {len(res)} | Buy months: {buys} | Sell months: {sells}')

In [ ]:
f = res.iloc[-1]
eq = res['pf']
ddr = (eq.cummax()-eq)/eq.cummax(); maxdd = ddr.max()*100
m = eq.pct_change().dropna(); m = m[m.index>=12]
sh = (m.mean()/m.std())*np.sqrt(12) if m.std()>0 else 0
p = m[m>0].sum(); ne = m[m<0].abs().sum(); pf = p/ne if ne>0 else float('inf')
ann = (1+(f['pf']/f['inj']-1))**(12/len(res))-1
cal = ann/(maxdd/100) if maxdd>0 else 0
btc_r = (f['bv']/(len(res)*20)-1)*100
bnb_r = (f['nv']/(len(res)*10)-1)*100
print('='*72)
print('PORTFOLIO BOT v2 — Monthly DCA + 2x ATH Sell')
print('='*72)
print(f'Period:       {res["date"].min().strftime("%Y-%m")} -> {res["date"].max().strftime("%Y-%m")} ({len(res)} mo)')
print(f'Injection:    $30/mo = ${f["inj"]:.0f} total')
print(f'Strategy:     Buy $20 BTC + $10 BNB every month; sell 35% at 2x ATH')
print()
print('--- ASSET BREAKDOWN ---')
print(f'BTC:          ${f["bv"]:.2f} ({f["bh"]:.6f} @ {f["bc"]:.0f})')
print(f'BTC return:   {btc_r:.1f}%')
print(f'BNB:          ${f["nv"]:.2f} ({f["nh"]:.6f} @ {f["nc"]:.0f})')
print(f'BNB return:   {bnb_r:.1f}%')
print(f'USDT reserve: ${f["usdt"]:.2f}')
print()
print('--- PORTFOLIO ---')
print(f'Portfolio:    ${f["pf"]:.2f}')
print(f'Total P&L:    ${f["pf"]-f["inj"]:.2f}')
print(f'Return:       {(f["pf"]/f["inj"]-1)*100:.2f}%')
print(f'Max DD:       {maxdd:.2f}%')
print(f'Sharpe:       {sh:.2f}')
print(f'PF:           {pf:.2f}')
print(f'Calmar:       {cal:.2f}')

In [ ]:
# Dashboard
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(5, 2, height_ratios=[3, 1, 1.2, 1.2, 1.2])
ax = fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['inj'], res['pf'],
    where=res['pf']>=res['inj'], color='green', alpha=0.12, label='Profit')
ax.fill_between(res['date'], res['pf'], res['inj'],
    where=res['pf']<res['inj'], color='red', alpha=0.12, label='Loss')
ax.plot(res['date'], res['inj'], color='gray', lw=1.5, ls='--', label='Injected ($30/mo)')
ax.plot(res['date'], res['pf'], color='#2563eb', lw=2, label='Portfolio')
sells = res[res['act'].str.contains('sell', na=False)]
ax.scatter(sells['date'], sells['pf'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 35%')
ax.set_title('Portfolio Bot v2 — Monthly DCA + 35% Sell at 2x ATH', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=8, ncol=3); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, ddr*100, color='#dc2626', alpha=0.35)
ax.plot(res['date'], ddr*100, color='#dc2626', lw=0.8)
ax.axhline(y=maxdd, color='#991b1b', ls=':', lw=1, label=f'Max DD: {maxdd:.1f}%')
ax.set_ylabel('DD (%)'); ax.set_ylim(bottom=0); ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[2, :])
ax.stackplot(res['date'], res['bv'], res['nv'], res['usdt'],
    labels=['BTC','BNB','USDT'], colors=['#f59e0b','#8b5cf6','#10b981'], alpha=0.8)
ax.set_ylabel('Allocation'); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylim(bottom=0)
ax = fig.add_subplot(gs[3, 0])
ax.fill_between(res['date'], 0, res['bv'], color='#f59e0b', alpha=0.3)
ax.plot(res['date'], res['bv'], color='#d97706', lw=1.5)
ax.set_ylabel('BTC Value'); ax.set_title('BTC'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[3, 1])
ax.fill_between(res['date'], 0, res['nv'], color='#8b5cf6', alpha=0.3)
ax.plot(res['date'], res['nv'], color='#7c3aed', lw=1.5)
ax.set_ylabel('BNB Value'); ax.set_title('BNB'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[4, 0])
ax.fill_between(res['date'], 0, res['usdt'], color='#10b981', alpha=0.3)
ax.plot(res['date'], res['usdt'], color='#059669', lw=1.5)
ax.set_ylabel('USDT'); ax.set_title('Cash Reserve'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[4, 1])
ax.plot(res['date'], res['bc'], color='#f59e0b', lw=1, alpha=0.7, label='BTC')
axb = ax.twinx(); axb.plot(res['date'], res['nc'], color='#8b5cf6', lw=1, alpha=0.7, label='BNB')
ax.set_ylabel('BTC'); axb.set_ylabel('BNB'); ax.set_title('Prices')
ax.grid(True, alpha=0.25); ax.legend(loc='upper left', fontsize=8); axb.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-v2.png', dpi=150, bbox_inches='tight')
print('Saved dca-bt-buy-bot-portfolio-v2.png')
plt.show()

In [ ]:
# Cashflow table
cf = []
for i, row in res.iterrows():
    bs = 0; ns = 0
    if i > 0:
        bd = row['bh'] - res.iloc[i-1]['bh']
        if bd < -0.000001: bs = abs(bd)*(row['bc']+res.iloc[i-1]['bc'])/2
        nd = row['nh'] - res.iloc[i-1]['nh']
        if nd < -0.000001: ns = abs(nd)*(row['nc']+res.iloc[i-1]['nc'])/2
    cf.append({'d': row['date'], 'bs': bs, 'ns': ns,
        'usdt': row['usdt'], 'pf': row['pf']})
cf = pd.DataFrame(cf)
total_sells_btc = cf['bs'].sum()
total_sells_bnb = cf['ns'].sum()
total_buys_btc = len(res) * 20
total_buys_bnb = len(res) * 10
print(f'Total BTC buys:  ${total_buys_btc:.0f}')
print(f'Total BNB buys:  ${total_buys_bnb:.0f}')
print(f'Total BTC sells: ${total_sells_btc:.0f}')
print(f'Total BNB sells: ${total_sells_bnb:.0f}')
print(f'Final reserve:   ${cf["usdt"].iloc[-1]:.0f}')
print(f'Final portfolio: ${cf["pf"].iloc[-1]:.0f}')
print()
print('SELL EVENTS:')
sells = res[res['act'].str.contains('sell', na=False)]
for _, r in sells.iterrows():
    print(f"  {r['date'].strftime('%Y-%m')}: {r['act']}")